In [1]:
# Cell 1 --- Imports, seed, output path
from pathlib import Path
import json, random, re
import numpy as np
import pandas as pd
from faker import Faker
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker("en_IN")
Faker.seed(SEED)

ROOT = Path.cwd()
OUT_DIR = ROOT / "Supa_dataset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Writing to:", OUT_DIR)
print("Seed:", SEED)

Writing to: C:\Users\avyaa\Supa_dataset
Seed: 42


In [2]:
# Cell 2 --- Helpers and distributions
def sample_price():
    p = np.random.lognormal(mean=9.4, sigma=0.8)
    return round(min(max(p, 50), 200000), 2)

def sample_popularity():
    return int(min(int(np.random.zipf(2.0)), 10_000))

def random_date_in_past(days=730):
    return pd.Timestamp.today(tz="UTC") - pd.to_timedelta(np.random.randint(0, days), unit="D")

STATUSES = ["pending","paid","shipped","delivered","cancelled","refunded"]
PAY_STAT = ["unpaid","paid","failed","refunded"]

_slug_rx = re.compile(r"[^a-z0-9]+")
def slugify(txt: str) -> str:
    s = txt.lower().strip()
    s = s.replace("&"," and ")
    s = _slug_rx.sub("-", s)
    return s.strip("-")

# TEXT[] literal builder - Critical fix for PostgreSQL array import
def pg_array_literal_from_list(items):
    """
    Convert a list[str] to a TEXT[] literal.
    Empty -> '{}'
    Non-empty -> {"a","b"} with backslashes and double-quotes escaped.
    """
    if not items:
        return "{}"
    
    def esc(s: str) -> str:
        s = s.replace("\\", "\\\\").replace('"', '\\"')
        return '"' + s + '"'
    
    return "{" + ",".join(esc(x) for x in items) + "}"

In [3]:
# Cell 3 --- Target counts (matching your research specs)
N_CATEGORIES = 50
N_CUSTOMERS = 25_000
N_PRODUCTS = 12_000
N_ORDERS = 250_000
N_ORDER_ITEMS = 500_000

print("Counts:", N_CATEGORIES, N_CUSTOMERS, N_PRODUCTS, N_ORDERS, N_ORDER_ITEMS)

Counts: 50 25000 12000 250000 500000


In [4]:
# Cell 4 --- Categories (nullable parent_id with Int64 to avoid BIGINT parse errors)
rows = []
for i in range(1, N_CATEGORIES + 1):
    name = fake.unique.word().title()
    parent = pd.NA if (i == 1 or np.random.rand() > 0.3) else np.random.randint(1, i)
    rows.append({
        "id": i,
        "name": name,
        "name_lc": name.lower(),
        "slug": slugify(name),
        "parent_id": parent,  # nullable
        "created_at": pd.Timestamp.utcnow(),
        "updated_at": pd.Timestamp.utcnow(),
    })

df_categories = pd.DataFrame(rows)
df_categories["id"] = df_categories["id"].astype("int64")
df_categories["parent_id"] = df_categories["parent_id"].astype("Int64")  # Critical: nullable integer
df_categories.to_csv(OUT_DIR / "categories.csv", index=False)
print(f"Categories: {len(df_categories)} rows")

Categories: 50 rows


In [6]:
# Cell 5 --- Customers (unique emails with realistic addresses) - FIXED
custs = []
for i in tqdm(range(1, N_CUSTOMERS + 1), desc="customers"):
    first = fake.first_name()
    last = fake.last_name()
    email = f"{first}.{last}.{i}@example.com".lower()
    
    # JSON address structure - using methods that work with en_IN locale
    address_json = {
        "line1": fake.street_address(),
        "line2": fake.building_number() + " " + fake.street_name() if np.random.rand() > 0.7 else None,
        "city": fake.city(),
        "state": fake.state(),
        "country": "India",
        "postal_code": fake.postcode()
    }
    
    custs.append({
        "id": i,
        "first_name": first,
        "last_name": last,
        "email": email,
        "email_lc": email.lower(),
        "phone": fake.phone_number(),
        "address": json.dumps(address_json, ensure_ascii=False),
        "address_line1": address_json["line1"],
        "address_line2": address_json.get("line2"),
        "city": address_json["city"],
        "state": address_json["state"],
        "country": address_json["country"],
        "postal_code": address_json["postal_code"],
        "is_active": bool(np.random.rand() > 0.03),
        "created_at": random_date_in_past(900),
        "updated_at": pd.Timestamp.utcnow(),
    })

df_customers = pd.DataFrame(custs)
df_customers["id"] = df_customers["id"].astype("int64")
df_customers.set_index("id", inplace=True)
df_customers.to_csv(OUT_DIR / "customers.csv", index=True, float_format="%.2f")
print(f"Customers: {len(df_customers)} rows")

customers: 100%|███████████████████████████████████████████████████████████████| 25000/25000 [00:05<00:00, 4317.50it/s]


Customers: 25000 rows


In [7]:
# Cell 6 --- Products (proper TEXT[] tags and JSONB attributes)
prods = []
tag_pool = ["new","sale","featured","limited","eco","refurbished","premium","bestseller"]
materials = ["cotton","leather","plastic","steel","wood","silicone","ceramic","aluminum"]
brands = ["TechCorp", "StylePlus", "EcoLiving", "ModernHome", "SportMax", "UrbanWear"]

for i in tqdm(range(1, N_PRODUCTS + 1), desc="products"):
    cat_id = np.random.randint(1, N_CATEGORIES + 1)
    brand = np.random.choice(brands)
    name = f"{fake.color_name()} {fake.word().title()}"
    sku = f"SKU-{cat_id:02d}-{i:06d}"
    
    attributes = {
        "brand": brand,
        "color": fake.color_name(),
        "weight_kg": round(abs(np.random.normal(1.2, 0.8)), 2),
        "material": np.random.choice(materials),
        "dimensions": {
            "length": round(np.random.uniform(10, 100), 1),
            "width": round(np.random.uniform(10, 100), 1),
            "height": round(np.random.uniform(5, 50), 1)
        }
    }
    
    # Generate tags with proper TEXT[] literal formatting
    chosen_tags = list(np.random.choice(tag_pool, size=np.random.randint(0,4), replace=False))
    tag_literal = pg_array_literal_from_list(chosen_tags)
    
    prods.append({
        "id": i,
        "sku": sku,
        "sku_lc": sku.lower(),
        "name": name,
        "name_lc": name.lower(),
        "description": fake.sentence(nb_words=12),
        "category_id": cat_id,
        "brand": brand,
        "brand_lc": brand.lower(),
        "price": sample_price(),
        "currency": "INR",
        "stock": int(max(np.random.poisson(lam=25), 0)),
        "popularity": sample_popularity(),
        "attributes": json.dumps(attributes, ensure_ascii=False),
        "tags": tag_literal,  # TEXT[] literal
        "is_active": bool(np.random.rand() > 0.05),
        "created_at": random_date_in_past(900),
        "updated_at": pd.Timestamp.utcnow(),
    })

df_products = pd.DataFrame(prods)
for col in ["id","category_id","stock","popularity"]:
    df_products[col] = df_products[col].astype("int64")
df_products.set_index("id", inplace=True)
df_products.to_csv(OUT_DIR / "products.csv", index=True, float_format="%.2f")
print(f"Products: {len(df_products)} rows")

products: 100%|████████████████████████████████████████████████████████████████| 12000/12000 [00:04<00:00, 2414.36it/s]


Products: 12000 rows


In [8]:
# Cell 7 --- Orders (JSON addresses and computed totals)
orders = []
for i in tqdm(range(1, N_ORDERS + 1), desc="orders"):
    cust_id = np.random.randint(1, N_CUSTOMERS + 1)
    when = random_date_in_past(730)
    status = np.random.choice(STATUSES, p=[0.25,0.30,0.15,0.15,0.10,0.05])
    payst = "paid" if status in ("paid","shipped","delivered") else np.random.choice(PAY_STAT)
    
    addr = {
        "line1": fake.street_address(),
        "city": fake.city(),
        "state": fake.state(),
        "country": "India",
        "postal_code": fake.postcode(),
    }
    
    orders.append({
        "id": i,
        "customer_id": cust_id,
        "order_date": when,
        "status": status,
        "payment_status": payst,
        "currency": "INR",
        "subtotal_amount": 0.00,  # will fill after rollup
        "tax_amount": 0.00,
        "shipping_fee": float(np.random.choice([0, 49, 99, 149])),
        "discount_amount": float(np.random.choice([0, 50, 100, 200, 500])),
        "total_amount": 0.00,  # will fill after rollup
        "coupon_code": np.random.choice([None, "WELCOME10","FESTIVE","FREESHIP"]),
        "shipping_address": json.dumps(addr, ensure_ascii=False),
        "billing_address": json.dumps(addr, ensure_ascii=False),
        "created_at": when,
        "updated_at": when,
    })

df_orders = pd.DataFrame(orders)
for col in ["id","customer_id"]:
    df_orders[col] = df_orders[col].astype("int64")
df_orders.set_index("id", inplace=True)
# Write provisional; will overwrite after rollup
df_orders.to_csv(OUT_DIR / "orders.csv", index=True, float_format="%.2f")
print(f"Orders: {len(df_orders)} rows (provisional)")

orders: 100%|████████████████████████████████████████████████████████████████| 250000/250000 [01:25<00:00, 2917.25it/s]


Orders: 250000 rows (provisional)


In [13]:
# Cell 8 --- Order items (NO DUPLICATES) - FIXED
oi_rows = []
used_combinations = set()  # Track (order_id, product_id) pairs

for oi_id in tqdm(range(1, N_ORDER_ITEMS + 1), desc="order_items"):
    # Keep trying until we find a unique (order_id, product_id) combination
    max_attempts = 100
    for attempt in range(max_attempts):
        order_id = np.random.randint(1, N_ORDERS + 1)
        product_id = np.random.randint(1, N_PRODUCTS + 1)
        
        combo = (order_id, product_id)
        if combo not in used_combinations:
            used_combinations.add(combo)
            break
    else:
        # If we couldn't find a unique combination after max_attempts,
        # skip this record (we'll have slightly fewer than target N_ORDER_ITEMS)
        print(f"Warning: Couldn't find unique combination for record {oi_id}, skipping")
        continue
    
    qty = max(1, int(np.random.poisson(lam=1.4)))
    price = float(df_products.loc[product_id, "price"])
    disc = float(np.random.choice([0, 0, 0, 50, 100]))
    total = max(round(price * qty - disc, 2), 0.0)
    
    oi_rows.append({
        "id": oi_id,
        "order_id": order_id,
        "product_id": product_id,
        "quantity": qty,
        "unit_price": price,
        "discount_amount": disc,
        "total_price": total,
        "category_id_denorm": int(df_products.loc[product_id, "category_id"]),
        "created_at": df_orders.loc[order_id, "order_date"],
    })

df_order_items = pd.DataFrame(oi_rows)
for col in ["id","order_id","product_id","quantity","category_id_denorm"]:
    df_order_items[col] = df_order_items[col].astype("int64")
df_order_items.to_csv(OUT_DIR / "order_items.csv", index=False, float_format="%.2f")
print(f"Order Items: {len(df_order_items)} rows (unique combinations)")

order_items: 100%|██████████████████████████████████████████████████████████| 500000/500000 [00:17<00:00, 28772.45it/s]


Order Items: 500000 rows (unique combinations)


In [12]:
# Cell 9 --- Roll up order totals (compute realistic financials) - FIXED
# Aggregate line items
group = df_order_items.groupby("order_id", as_index=True).agg(
    qty_sum = pd.NamedAgg(column="quantity", aggfunc="sum"),
    line_total = pd.NamedAgg(column="total_price", aggfunc=lambda s: float(np.round(s.sum(), 2))),
    line_disc = pd.NamedAgg(column="discount_amount", aggfunc=lambda s: float(np.round(s.sum(), 2))),
)

# Correct subtotal as sum(unit_price * qty) ignoring line-level discount
tmp = df_order_items.assign(ext=lambda df: np.round(df["unit_price"] * df["quantity"], 2))
line_subtotals = tmp.groupby("order_id")["ext"].sum().astype(float).round(2)
group["line_subtotal"] = line_subtotals

# Join into orders, compute tax and totals
orders_full = df_orders.join(group, how="left")
orders_full[["line_subtotal","line_total","line_disc"]] = orders_full[["line_subtotal","line_total","line_disc"]].fillna(0.0)

# Effective base = subtotal - line_disc - order_discount (non-negative)
base = (orders_full["line_subtotal"] - orders_full["line_disc"] - orders_full["discount_amount"]).clip(lower=0.0)
tax = (base * 0.18).round(2)

orders_full["subtotal_amount"] = orders_full["line_subtotal"].round(2)
orders_full["tax_amount"] = tax
orders_full["total_amount"] = (base + orders_full["shipping_fee"] + tax).round(2)

# CRITICAL FIX: Select only the original schema columns before writing CSV
schema_columns = [
    "id", "customer_id", "order_date", "status", "payment_status", "currency",
    "subtotal_amount", "tax_amount", "shipping_fee", "discount_amount", "total_amount",
    "coupon_code", "shipping_address", "billing_address", "created_at", "updated_at"
]

# Reset index to get 'id' as a column, then select only schema columns
orders_final = orders_full.reset_index()[schema_columns]
orders_final.to_csv(OUT_DIR / "orders.csv", index=False, float_format="%.2f")
print(f"Orders: {len(orders_final)} rows (final with computed totals)")

Orders: 250000 rows (final with computed totals)


In [14]:
# Cell 10 --- Manifest (reproducibility metadata)
manifest = {
    "seed": SEED,
    "counts": {
        "categories": len(df_categories),
        "customers": len(df_customers), 
        "products": len(df_products),
        "orders": len(df_orders),
        "order_items": len(df_order_items),
    },
    "total_records": len(df_categories) + len(df_customers) + len(df_products) + len(df_orders) + len(df_order_items),
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "schema_version": "supabase_v1",
    "benchmark_study": "free_tier_comparison_2025"
}

with open(OUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Dataset generation complete!")
print("Manifest written at:", OUT_DIR / "run_manifest.json")
print("\n=== IMPORT CHECKLIST ===")
print("Import order: categories → customers → products → orders → order_items")
print("CSV options: Header ON, Delimiter ',', Quote '\"', Escape '\"', Encoding UTF-8")
print("If partial load happens: TRUNCATE TABLE public.<name> RESTART IDENTITY CASCADE and re-import")

Dataset generation complete!
Manifest written at: C:\Users\avyaa\Supa_dataset\run_manifest.json

=== IMPORT CHECKLIST ===
Import order: categories → customers → products → orders → order_items
CSV options: Header ON, Delimiter ',', Quote '"', Escape '"', Encoding UTF-8
If partial load happens: TRUNCATE TABLE public.<name> RESTART IDENTITY CASCADE and re-import
